In [4]:
import cv2

# Load the image
img = cv2.imread("C:/Users/dpava/Desktop/mypic.jpg")

# Load the Haar Cascade classifier for face detection
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

# Detect faces in the image
faces = face_cascade.detectMultiScale(img, scaleFactor=1.1, minNeighbors=5)

# Blur each face detected in the image
for (x, y, w, h) in faces:
    # Define the region of interest as the face
    roi = img[y:y+h, x:x+w]
    
    # Apply Gaussian blur to the face
    blurred = cv2.GaussianBlur(roi, (23, 23), 30)
    
    # Replace the original face with the blurred face
    img[y:y+h, x:x+w] = blurred

# Display the blurred image
cv2.imshow("Blurred image", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
import heapq
import numpy as np
import cv2

class HuffmanCoding:
    def __init__(self):
        self.heap = []
        self.codes = {}
        self.reverse_mapping = {}

    class HeapNode:
        def __init__(self, pixel_intensity, freq):
            self.pixel_intensity = pixel_intensity
            self.freq = freq
            self.left = None
            self.right = None

        def __lt__(self, other):
            return self.freq < other.freq

        def __eq__(self, other):
            if(other == None):
                return False
            if(not isinstance(other, HeapNode)):
                return False
            return self.freq == other.freq

    def make_frequency_dict(self, image):
        frequency_dict = {}
        for pixel in image:
            if pixel in frequency_dict:
                frequency_dict[pixel] += 1
            else:
                frequency_dict[pixel] = 1
        return frequency_dict

    def make_heap(self, frequency_dict):
        for key in frequency_dict:
            node = self.HeapNode(key, frequency_dict[key])
            heapq.heappush(self.heap, node)

    def merge_nodes(self):
        while(len(self.heap)>1):
            node1 = heapq.heappop(self.heap)
            node2 = heapq.heappop(self.heap)

            merged = self.HeapNode(None, node1.freq + node2.freq)
            merged.left = node1
            merged.right = node2

            heapq.heappush(self.heap, merged)

    def make_codes_helper(self, root, current_code):
        if(root == None):
            return

        if(root.pixel_intensity != None):
            self.codes[root.pixel_intensity] = current_code
            self.reverse_mapping[current_code] = root.pixel_intensity
            return

        self.make_codes_helper(root.left, current_code + "0")
        self.make_codes_helper(root.right, current_code + "1")

    def make_codes(self):
        root = heapq.heappop(self.heap)
        current_code = ""
        self.make_codes_helper(root, current_code)

    def get_encoded_image(self, image):
        encoded_image = ""
        for pixel in image:
            encoded_image += self.codes[pixel]
        return encoded_image

    def pad_encoded_image(self, encoded_image):
        extra_padding = 8 - len(encoded_image) % 8
        for i in range(extra_padding):
            encoded_image += "0"

        padded_info = "{0:08b}".format(extra_padding)
        encoded_image = padded_info + encoded_image
        return encoded_image

    def get_byte_array(self, padded_encoded_image):
        if(len(padded_encoded_image) % 8 != 0):
            print("Encoded image not padded properly")
            exit(0)

        b = bytearray()
        for i in range(0, len(padded_encoded_image), 8):
            byte = padded_encoded_image[i:i+8]
            b.append(int(byte, 2))
        return b

    def compress(self, image_path):
        image = cv2.imread(image_path, 0)
        image = np.array(image).flatten()

        frequency_dict = self.make_frequency_dict(image)
        self.make_heap(frequency_dict)
        self.merge_nodes()
        self.make_codes()

        encoded_image = self.get_encoded_image(image)
        padded_encoded_image = self.pad_encoded_image(encoded_image)

        byte_array = self.get_byte_array(padded_encoded_image)
        return byte_array

    def remove_padding(self, padded_encoded_image):
        padded_info = padded_encoded_image[:8]
        extra_padding = int(padded_info, 2)

        padded_encoded_image = padded